In [1]:
import os
import numpy as np
from scipy.linalg import eigh_tridiagonal
import scipy.sparse as sp
import itertools
import math
from joblib import dump
import time
from numba import njit, prange

# --- 0. Global Physical Constants ---
CONV_EV_TO_K = 11604.5 
t_phys_eV = 0.3         

# ---------------------------------------------------------
# 1. Basis & Matrix Generation (SINGLE SPIN ONLY)
# ---------------------------------------------------------
def generate_single_spin_basis(L, N_p):
    combos = itertools.combinations(range(L), N_p)
    basis = np.array([sum(1 << b for b in combo) for combo in combos], dtype=np.int64)
    state_to_idx = {state: idx for idx, state in enumerate(basis)}
    return basis, state_to_idx

def get_single_spin_matrix(Lx, Ly, t, tp, basis, state_to_idx, bc='PBC'):
    L = Lx * Ly
    dim = len(basis)
    rows, cols, data = [], [], []
    
    edges_nn, edges_nnn = set(), set()
    is_1d = (Lx == 1 or Ly == 1)
    
    for i in range(L):
        if is_1d:
            if bc == 'PBC' or i + 1 < L: edges_nn.add(tuple(sorted((i, (i + 1) % L))))
            if tp != 0.0 and (bc == 'PBC' or i + 2 < L): edges_nnn.add(tuple(sorted((i, (i + 2) % L))))
        else:
            x, y = i % Lx, i // Lx
            nx1, ny1 = (x + 1) % Lx, y
            nx2, ny2 = x, (y + 1) % Ly
            
            if bc == 'PBC' or (x + 1 < Lx): edges_nn.add(tuple(sorted((i, ny1 * Lx + nx1))))
            if bc == 'PBC' or (y + 1 < Ly): edges_nn.add(tuple(sorted((i, ny2 * Lx + nx2))))
                
            if tp != 0.0:
                nx3, ny3 = (x + 1) % Lx, (y + 1) % Ly
                nx4, ny4 = (x + 1) % Lx, (y - 1) % Ly
                if bc == 'PBC' or (x + 1 < Lx and y + 1 < Ly): edges_nnn.add(tuple(sorted((i, ny3 * Lx + nx3))))
                if bc == 'PBC' or (x + 1 < Lx and y - 1 >= 0): edges_nnn.add(tuple(sorted((i, ny4 * Lx + nx4))))
                
    edges_nn = [e for e in edges_nn if e[0] != e[1]]
    if tp != 0.0: edges_nnn = [e for e in edges_nnn if e[0] != e[1]]

    for idx, s in enumerate(basis):
        hopping_list = [(edges_nn, t)]
        if tp != 0.0: hopping_list.append((edges_nnn, tp))

        for edges, hop_val in hopping_list:
            if hop_val == 0.0: continue
            for i, j in edges:
                if ((s >> i) & 1) == 1 and ((s >> j) & 1) == 0:
                    s_new = s ^ (1 << i) ^ (1 << j)
                    idx_new = state_to_idx[s_new]
                    m1, m2 = min(i, j), max(i, j)
                    count = (s & ((1 << m2) - (1 << (m1 + 1)))).bit_count()
                    val = -hop_val * (1 if count % 2 == 0 else -1)
                    
                    rows.append(idx_new); cols.append(idx); data.append(val)
                    rows.append(idx); cols.append(idx_new); data.append(val)

    return sp.coo_matrix((data, (rows, cols)), shape=(dim, dim)).tocsr()
    
# --- Massively fast pre-computation of interaction energies ---
@njit(parallel=True, fastmath=True)
def build_U_diag_fast(U, up_basis, dn_basis):
    N_up = len(up_basis)
    N_dn = len(dn_basis)
    U_diag = np.zeros((N_up, N_dn), dtype=np.float64)
    for i in prange(N_up):
        u_state = up_basis[i]
        for j in range(N_dn):
            d_state = dn_basis[j]
            # Bitwise trick to count set bits (double occupancies)
            tmp = u_state & d_state
            c = 0
            while tmp > 0:
                tmp &= tmp - 1
                c += 1
            U_diag[i, j] = U * c
    return U_diag

# ---------------------------------------------------------
# 2. Spectral Precomputation (Tensor Network Mat-Vec)
# ---------------------------------------------------------
@njit(parallel=True, fastmath=True)
def tensor_spmv(H_up_data, H_up_indices, H_up_indptr,
                H_dn_data, H_dn_indices, H_dn_indptr,
                U_diag, V, W):
    """
    Computes W = H_up * V + V * H_dn^T + U_diag * V with perfect cache locality.
    """
    N_up = V.shape[0]
    N_dn = V.shape[1]

    for u in prange(N_up):
        # 1. Diagonal U term + H_dn hopping (acts purely on down-spin index d)
        for d in range(N_dn):
            val = U_diag[u, d] * V[u, d]
            for p in range(H_dn_indptr[d], H_dn_indptr[d+1]):
                d_prime = H_dn_indices[p]
                val += H_dn_data[p] * V[u, d_prime]
            W[u, d] = val

        # 2. H_up hopping (acts purely on up-spin index u)
        for p in range(H_up_indptr[u], H_up_indptr[u+1]):
            u_prime = H_up_indices[p]
            hop = H_up_data[p]
            # Fast inner loop over contiguous memory
            for d in range(N_dn):
                W[u, d] += hop * V[u_prime, d]

def _compute_single_sample(H_up, H_dn, U_diag, N_tot, N_up, N_dn_dim, steps):
    v_1d = np.random.choice([-1.0, 1.0], size=N_tot)
    v_1d /= np.sqrt(N_tot)
    
    alpha, beta = np.zeros(steps), np.zeros(steps - 1)
    v_prev_1d = None
    v_curr_1d = v_1d
    
    # Pre-allocate W as a 1D array to avoid reallocation
    w_1d = np.zeros(N_tot, dtype=np.float64)
    
    for j in range(steps):
        # Create 2D views without copying memory
        V_2d = v_curr_1d.reshape((N_up, N_dn_dim))
        W_2d = w_1d.reshape((N_up, N_dn_dim))
        
        # --- Tensor Mat-Vec ---
        tensor_spmv(H_up.data, H_up.indices, H_up.indptr,
                    H_dn.data, H_dn.indices, H_dn.indptr,
                    U_diag, V_2d, W_2d)
        
        if v_prev_1d is not None: w_1d -= beta[j-1] * v_prev_1d
        alpha[j] = np.dot(v_curr_1d, w_1d)
        w_1d -= alpha[j] * v_curr_1d
        
        if j < steps - 1:
            beta[j] = np.linalg.norm(w_1d)
            if beta[j] < 1e-12: 
                alpha, beta = alpha[:j+1], beta[:j]; break
            v_prev_1d, v_curr_1d = v_curr_1d, w_1d / beta[j]
            
    evals, evecs = eigh_tridiagonal(alpha, beta)
    return evals, evecs[0, :]**2

def compute_sector_spectra_batched(H_up, H_dn, U_diag, Lx, Ly, N_up, N_dn, total_sample, sample_unit, steps, cache_dir):
    N_up_dim = H_up.shape[0]
    N_dn_dim = H_dn.shape[0]
    N_tot = N_up_dim * N_dn_dim
    
    # Exact Diagonalization Fallback for small matrices
    if N_tot <= 2000: 
        print(f"      -> Exact Diagonalization used (Dim: {N_tot} <= 2000).")
        H_full = sp.kron(H_up, sp.eye(N_dn_dim)) + sp.kron(sp.eye(N_up_dim), H_dn) + sp.diags(U_diag.ravel())
        evals = np.linalg.eigvalsh(H_full.toarray())
        return 'exact', evals    

    sector_file = os.path.join(cache_dir, f"sector_{Lx}x{Ly}_up{N_up}_dn{N_dn}.joblib")
    
    # Always start fresh (load logic removed)
    accumulated_samples = []
        
    while len(accumulated_samples) < total_sample:
        remaining = total_sample - len(accumulated_samples)
        batch_size = min(sample_unit, remaining)
        
        batch_start = time.time()
        new_samples = []
        max_allowed_steps = min(steps, N_tot)
        
        for _ in range(batch_size):
            new_samples.append(_compute_single_sample(H_up, H_dn, U_diag, N_tot, N_up_dim, N_dn_dim, max_allowed_steps))
        
        accumulated_samples.extend(new_samples)
        
        # Save progress sector-by-sector
        dump(accumulated_samples, sector_file)
        
        batch_dur = time.time() - batch_start
        steps_taken = [len(samp[0]) for samp in new_samples]
        
        if batch_size == 1:
            print(f"      -> Sample {len(accumulated_samples)}/{total_sample} | Time: {batch_dur:.2f}s | Lanczos Steps: {steps_taken[0]} / {max_allowed_steps}")
        else:
            avg_steps = sum(steps_taken) / len(steps_taken)
            print(f"      -> Samples: {len(accumulated_samples):>3}/{total_sample} | Time: {batch_dur:.2f}s | Avg Lanczos Steps: {avg_steps:.1f} / {max_allowed_steps}")
        
    return 'ftlm', N_tot, accumulated_samples

def _compute_single_sector(Lx, Ly, t, tp, U, N_up, N_dn, total_sample, sample_unit, steps, bc, cache_dir):
    gen_start = time.time()
    
    # 1. Generate single-spin basis
    up_basis, up_state_to_idx = generate_single_spin_basis(Lx*Ly, N_up)
    dn_basis, dn_state_to_idx = generate_single_spin_basis(Lx*Ly, N_dn)
    
    # 2. Generate single-spin matrices
    H_up = get_single_spin_matrix(Lx, Ly, t, tp, up_basis, up_state_to_idx, bc)
    H_dn = get_single_spin_matrix(Lx, Ly, t, tp, dn_basis, dn_state_to_idx, bc)
    
    # 3. Generate Interaction Matrix
    U_diag = build_U_diag_fast(U, up_basis, dn_basis)
    
    gen_dur = time.time() - gen_start
    N_tot = len(up_basis) * len(dn_basis)
    print(f"    [Dim: {N_tot:,}] Tensor Matrices generated in {gen_dur:.3f}s")
    
    res = compute_sector_spectra_batched(H_up, H_dn, U_diag, Lx, Ly, N_up, N_dn, total_sample, sample_unit, steps, cache_dir)
    return (N_up, N_dn), res

def precompute_all_sectors(Lx, Ly, t, tp, U, total_sample=400, sample_unit=100, steps=40, bc='PBC'):
    L = Lx * Ly
    cache_dir = f"cache_lanczos_{Lx}x{Ly}_t{t}_tp{tp}_U{U}"
    os.makedirs(cache_dir, exist_ok=True)
    
    pairs = [(u, d) for u in range(L+1) for d in range(u+1)]
    pairs.sort(key=lambda x: math.comb(L, x[0])*math.comb(L, x[1]), reverse=True)
    
    spectra = {}
    for u, d in pairs:
        print(f"\n  -> Computing Sector: Up: {u}, Dn: {d}")
        _, res = _compute_single_sector(Lx, Ly, t, tp, U, u, d, total_sample, sample_unit, steps, bc, cache_dir)
        spectra[(u, d)] = res
        if u != d: spectra[(d, u)] = res
        
    return spectra

# ---------------------------------------------------------
# 3. Main Execution (Data Generation Only)
# ---------------------------------------------------------
if __name__ == "__main__":
    total_start_time = time.time()

    lattice_configs = [(4,4)] 
    t, tp, U = 1.0, -0.2, 8
    
    print("NOTE: Numba will compile the parallel kernel during the first sector.")

    for (Lx, Ly) in lattice_configs:
        L = Lx * Ly
        print(f"\n=======================================================")
        print(f"Generating Data for Lattice {Lx}x{Ly} (L={L})")
        print(f"=======================================================")
        
        TARGET_SAMPLES = 10
        if L >= 16:   SAMPLE_UNIT = 10   
        elif L >= 14: SAMPLE_UNIT = 50   
        elif L >= 12: SAMPLE_UNIT = 100  
        else:         SAMPLE_UNIT = 400  
            
        print(f"--> Target Samples: {TARGET_SAMPLES} (Batched in chunks of {SAMPLE_UNIT})")

        run_id = 3 # Change this number for each new simulation you want to run
        full_spectra_file = f"full_spectra_dict_{Lx}x{Ly}_t{t}_tp{tp}_U{U}_run{run_id}.joblib"
        
        print(f"--> Computing spectra from scratch...")
        spectra_cache_unbiased = precompute_all_sectors(
            Lx, Ly, t, tp, U, 
            total_sample=TARGET_SAMPLES, 
            sample_unit=SAMPLE_UNIT, 
            bc='PBC'
        )
        
        dump(spectra_cache_unbiased, full_spectra_file)
        print(f"--> Saved full spectra dictionary to {full_spectra_file}.")

    total_elapsed_sec = time.time() - total_start_time
    print(f"\n=======================================================")
    print(f"TOTAL GENERATION TIME: {total_elapsed_sec:.2f} seconds ({total_elapsed_sec/60:.2f} minutes)")
    print(f"=======================================================\n")

NOTE: Numba will compile the parallel kernel during the first sector.

Generating Data for Lattice 4x4 (L=16)
--> Target Samples: 10 (Batched in chunks of 10)
--> Computing spectra from scratch...

  -> Computing Sector: Up: 8, Dn: 8


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


    [Dim: 165,636,900] Tensor Matrices generated in 2.596s
      -> Samples:  10/10 | Time: 1047.03s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 8, Dn: 7
    [Dim: 147,232,800] Tensor Matrices generated in 0.741s
      -> Samples:  10/10 | Time: 980.26s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 9, Dn: 8
    [Dim: 147,232,800] Tensor Matrices generated in 0.822s
      -> Samples:  10/10 | Time: 948.84s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 7, Dn: 7
    [Dim: 130,873,600] Tensor Matrices generated in 0.679s
      -> Samples:  10/10 | Time: 909.40s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 9, Dn: 7
    [Dim: 130,873,600] Tensor Matrices generated in 0.690s
      -> Samples:  10/10 | Time: 917.22s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 9, Dn: 9
    [Dim: 130,873,600] Tensor Matrices generated in 0.719s
      -> Samples:  10/10 | Time: 900.57s | Avg Lanczos Steps: 40.0 / 40

  -> Computing Sector: Up: 